In [ ]:
# Parameters -- edit these before running
REPO_URL    = "https://github.com/Dev252001/ham10000-lesion-eval.git"
REPO_BRANCH = "main"  # set to another branch to test before merging
SEED        = 42
SPLIT_MODE  = "lesion"  # "image" is for the leakage comparison
MODELS      = ["resnet18", "efficientnet_b0"]

In [ ]:
# Setup: mount Drive, clone/pull repo, install extras
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import subprocess, sys
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ham10000_project")
REPO_DIR  = DRIVE_DIR / "ham10000-lesion-eval"
CKPT_DIR  = DRIVE_DIR / "checkpoints"
DATA_DIR  = Path("/content/ham10000_data")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / "src"))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib", "Pillow", "grad-cam", "kaggle"],
    check=True,
)

In [ ]:
# Copy dataset from Drive cache
import shutil
DATA_CACHE = DRIVE_DIR / "ham10000_raw"
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
shutil.copytree(str(DATA_CACHE), str(DATA_DIR))
print(f"Data at {DATA_DIR}")

In [ ]:
# Shared setup: data + split (same for all models)
import torch
from skinlesion.data import load_metadata, HAMDataset
from skinlesion.splits import make_split, split_manifest, manifest_sha256
from skinlesion.transforms import train_transform, eval_transform
from skinlesion.models import build_model, compute_class_weights
from skinlesion.config import TrainConfig
from skinlesion.train import load_or_train
from skinlesion.evaluate import final_test_evaluation
from torch.utils.data import DataLoader

df = load_metadata(DATA_DIR)
train_df, val_df, test_df = make_split(df, split_mode=SPLIT_MODE, seed=SEED)
manifest = split_manifest(train_df, val_df, test_df, split_mode=SPLIT_MODE, seed=SEED)
sha = manifest_sha256(manifest)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

class_weights = compute_class_weights(train_df)

In [ ]:
# --- resnet18 ---
# Each model is a self-contained cell group; a session cut-off loses at most one model.
model_name = "resnet18"
cfg_r18 = TrainConfig(model_name=model_name, split_mode=SPLIT_MODE, seed=SEED, manifest_sha256=sha)

train_ds_r18 = HAMDataset(train_df, train_transform(cfg_r18.image_size))
val_ds_r18   = HAMDataset(val_df,   eval_transform(cfg_r18.image_size))
train_loader_r18 = DataLoader(train_ds_r18, batch_size=cfg_r18.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_r18   = DataLoader(val_ds_r18,   batch_size=cfg_r18.batch_size, shuffle=False, num_workers=2, pin_memory=True)

model_r18 = build_model(model_name, pretrained=True)
model_r18, history_r18, from_cache_r18 = load_or_train(
    model=model_r18,
    train_loader=train_loader_r18,
    val_loader=val_loader_r18,
    cfg=cfg_r18,
    class_weights=class_weights,
    device=device,
    ckpt_dir=CKPT_DIR,
)
print(f"resnet18: best_epoch={history_r18.best_epoch}  train_time={history_r18.train_time_s:.1f}s  from_cache={from_cache_r18}")

In [ ]:
# resnet18 -- final test evaluation
test_ds_r18     = HAMDataset(test_df, eval_transform(cfg_r18.image_size))
test_loader_r18 = DataLoader(test_ds_r18, batch_size=cfg_r18.batch_size, shuffle=False, num_workers=2)

metrics_r18, _, _ = final_test_evaluation(
    model=model_r18,
    test_df=test_df,
    test_loader=test_loader_r18,
    cfg=cfg_r18,
    history=history_r18,
    manifest_sha256=sha,
    repo_dir=REPO_DIR,
    outputs_dir=REPO_DIR / "outputs",
    device=device,
)
print(f"resnet18 macro-F1: {metrics_r18['metrics']['macro_f1']:.4f}")

In [ ]:
# --- efficientnet_b0 ---
model_name = "efficientnet_b0"
cfg_eff = TrainConfig(model_name=model_name, split_mode=SPLIT_MODE, seed=SEED, manifest_sha256=sha)

train_ds_eff = HAMDataset(train_df, train_transform(cfg_eff.image_size))
val_ds_eff   = HAMDataset(val_df,   eval_transform(cfg_eff.image_size))
train_loader_eff = DataLoader(train_ds_eff, batch_size=cfg_eff.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_eff   = DataLoader(val_ds_eff,   batch_size=cfg_eff.batch_size, shuffle=False, num_workers=2, pin_memory=True)

model_eff = build_model(model_name, pretrained=True)
model_eff, history_eff, from_cache_eff = load_or_train(
    model=model_eff,
    train_loader=train_loader_eff,
    val_loader=val_loader_eff,
    cfg=cfg_eff,
    class_weights=class_weights,
    device=device,
    ckpt_dir=CKPT_DIR,
)
print(f"efficientnet_b0: best_epoch={history_eff.best_epoch}  train_time={history_eff.train_time_s:.1f}s  from_cache={from_cache_eff}")

In [ ]:
# efficientnet_b0 -- final test evaluation
test_ds_eff     = HAMDataset(test_df, eval_transform(cfg_eff.image_size))
test_loader_eff = DataLoader(test_ds_eff, batch_size=cfg_eff.batch_size, shuffle=False, num_workers=2)

metrics_eff, _, _ = final_test_evaluation(
    model=model_eff,
    test_df=test_df,
    test_loader=test_loader_eff,
    cfg=cfg_eff,
    history=history_eff,
    manifest_sha256=sha,
    repo_dir=REPO_DIR,
    outputs_dir=REPO_DIR / "outputs",
    device=device,
)
print(f"efficientnet_b0 macro-F1: {metrics_eff['metrics']['macro_f1']:.4f}")

In [ ]:
# Package outputs and download
from skinlesion.export import make_outputs_zip
from google.colab import files

zip_path = DRIVE_DIR / "outputs_03.zip"
make_outputs_zip(REPO_DIR, zip_path)
files.download(str(zip_path))
print("Unzip at the root of your local clone, then commit.")